In [1]:
from pyhive import hive

host_name = "localhost"
port = 10000  #default is 10000
user = "root" # user name mysql
password = "password" # pass mysql
database="default"

def hiveconnection(host_name, port, user,password, database):
    conn = hive.Connection(host=host_name, port=port, username=user,password=password,
                           database=database, auth='CUSTOM')
    cur = conn.cursor()
    cur.execute('select name  from demo2 return limit 2')
    result = cur.fetchall()

    return result


In [2]:
output = hiveconnection(host_name, port, user,password, database)
print(output)

[('chantana',), ('ploy',)]


In [4]:
import subprocess
import os
output = subprocess.run(
    ["/home/hadoop/hadoop/bin/hadoop", "classpath", "--glob"], 
    capture_output = True,
    text = True
)

os.environ["CLASSPATH"] = output.stdout

In [12]:
%%file /home/hadoop/pyarrow_hdfs.py

 
import pyarrow as pa
from pyarrow import fs

import subprocess
import os
output = subprocess.run(
    ["/home/hadoop/hadoop/bin/hadoop", "classpath", "--glob"], 
    capture_output = True,
    text = True
)

os.environ["CLASSPATH"] = output.stdout



table_content = "row1_column1\trow1_column_2\nrow2_column1\trow2_column2".encode()
filename =  "/my_first_hdfs_table.tsv"

hdfs = fs.HadoopFileSystem("localhost", 9000) # Use fs.HadoopFileSystem

with hdfs.open_output_stream(filename) as of:   
   print('opened')
   of.write(table_content)

of.close()

Overwriting /home/hadoop/pyarrow_hdfs.py


run the following in terminal


python3 /home/hadoop/pyarrow_hdfs.py


Test with hive connection

In [15]:
user='root'
password = 'password'
host_name = 'localhost'

conn = hive.Connection(host=host_name, port=port, username=user,password=password,
                           database=database, auth='CUSTOM')
hive_cur = conn.cursor()

# If you would like to convert Text (or another format) based Hive table, you could use a trick like this:
# Text file-based external Hive Table
# assume server1 is your hostname or you can use localhost
hdfs_loc = "hdfs://localhost:9000/my_first_hdfs_hive_table.tsv"
tb_name = "my_hdfs_external_table"
table_body  = '(name String, salary String) '
db_name = "database"

#  Creating external HDFS based Hive table "
create_tb = ('CREATE TABLE IF NOT EXISTS %s  %s '
                       'ROW FORMAT DELIMITED FIELDS TERMINATED BY \'\\t\' '
                       'LOCATION \'%s\'') % (  tb_name, table_body, hdfs_loc)

create_tb2 = 'CREATE TABLE IF NOT EXISTS employee ( eid int, name String, salary String, destination String) COMMENT \'Employee details\' ROW FORMAT DELIMITED FIELDS TERMINATED BY \'\\t\' LINES TERMINATED BY  \'\\n\' STORED AS TEXTFILE'
print(create_tb)
print(create_tb2)


hive_cur.execute(create_tb)

#  Creating internal Parquet table with the same structure
parquet_tb_name = "my_parquete_table"

table_format = "PARQUET"
create_tb = ('CREATE TABLE IF NOT EXISTS %s %s STORED AS %s') % (parquet_tb_name, table_body, table_format)
print(create_tb)
hive_cur.execute(create_tb)


# Converting tables
query = ("INSERT OVERWRITE TABLE %s SELECT * FROM %s") % ( parquet_tb_name,  tb_name)
print(query)
hive_cur.execute(query)

query = ("SELECT * FROM %s") % ( parquet_tb_name)
print(query)
result = hive_cur.execute(query)
result

CREATE TABLE IF NOT EXISTS my_hdfs_external_table  (name String, salary String)  ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t' LOCATION 'hdfs://localhost:9000/my_first_hdfs_hive_table.tsv'
CREATE TABLE IF NOT EXISTS employee ( eid int, name String, salary String, destination String) COMMENT 'Employee details' ROW FORMAT DELIMITED FIELDS TERMINATED BY '\t' LINES TERMINATED BY  '\n' STORED AS TEXTFILE
CREATE TABLE IF NOT EXISTS my_parquete_table (name String, salary String)  STORED AS PARQUET
INSERT OVERWRITE TABLE my_parquete_table SELECT * FROM my_hdfs_external_table
SELECT * FROM my_parquete_table
